# Quantum tunnelling

*quantum · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/quantum/tunneling)

A Gaussian wave packet hits a rectangular barrier. The time-dependent Schrödinger equation is solved with the split-step Fourier method: the potential is a diagonal operator in $x$-space and the kinetic energy is a diagonal operator in $k$-space, so each half of a time step is just an elementwise tensor multiply, with torch.fft switching between the two bases.

**Vector form**

$$i\,\partial_t\psi = \Big(-\tfrac12\,\partial_x^2 + V(x)\Big)\psi,\qquad \psi(t+\Delta t) \approx e^{-iV\Delta t/2}\,\mathcal F^{-1} e^{-ik^2\Delta t/2}\,\mathcal F\,e^{-iV\Delta t/2}\,\psi(t)$$

**Index (tensor) form**

$$\psi_x \leftarrow P_x\,\psi_x,\qquad \tilde\psi_k = F_{kx}\psi_x,\qquad \tilde\psi_k \leftarrow K_k\,\tilde\psi_k,\qquad \psi_x = F^{-1}_{xk}\tilde\psi_k,\qquad \psi_x \leftarrow P_x\,\psi_x$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\psi_x$ | `psi` | (N,) complex | wavefunction on the periodic grid $x$ |
| $P_x = e^{-iV_x\Delta t/2}$ | `potential_phase(V, dt)` | (N,) complex | half-step potential operator: diagonal in $x$, applied as psi * P |
| $K_k = e^{-ik^2\Delta t/2}$ | `kinetic_phase(k, dt)` | (N,) complex | full-step kinetic operator: diagonal in $k$, applied as psi_k * K |
| $F_{kx}\psi_x$ | `torch.fft.fft(psi)` | (N,) complex | change of basis $x \to k$ (a fast matrix product, never built explicitly) |
| $P_T$ | `probability(psi[x > a/2], dx)` | () | transmission: probability found to the right of the barrier |

Units $\hbar = m = 1$, packet energy $E \approx k_0^2/2$. A diagonal operator $D_{xy} = d_x\,\delta_{xy}$ acting on $\psi$ is $D_{xy}\psi_y = d_x\psi_x$ — no matrix needed, just an elementwise multiply. The Strang splitting $P\cdot K\cdot P$ is second-order accurate in $\Delta t$ and exactly unitary, so the norm stays 1 to round-off. The simulated time is chosen automatically ($45/k_0$) so the packet centre travels from $x = -25$ to about $x = +20$. For $E < V_0$ a classical particle would always bounce back; the transmitted lump on the right is tunnelling. The plane-wave transmission coefficient $T(k_0)$ printed for comparison ignores the packet's spread of momenta (and uses the barrier width as it lands on the grid); a wider packet (larger $\sigma_0$) has a sharper momentum and matches it more closely.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/quantum/tunneling/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import probability


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    k0, V0, width = params["k0"], params["V0"], params["width"]
    x, k, dx = tensors.grid(2048, 200.0, dev)
    V = tensors.barrier(x, V0, width)
    psi0 = tensors.gaussian_packet(x, x0=-25.0, sigma=params["sigma"], k0=k0, dx=dx)

    times, psi = solve.solve(psi0, V, k, t_max=45.0 / k0)
    p_trans = solve.transmission(psi, x, width, dx)

    E = k0**2 / 2
    T_plane = solve.plane_wave_transmission(E, V0, tensors.grid_width(x, width, dx))
    print(f"packet energy E = {E:.3f}, barrier V0 = {V0:.3f}")
    print(f"transmission P_T = {p_trans[-1]:.4f}   plane-wave T(k0) = {T_plane:.4f}")
    print(f"norm at t_max: {probability(psi[-1], dx):.8f}")
    return plot.animate(x, times, psi, V, E, p_trans, out_dir / "tunneling.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/quantum/tunneling/equation.py
r"""Physics → tensor translation.

Time-dependent Schrödinger equation in 1D (units ħ = m = 1):

    i \partial_t \psi = (\hat T + \hat V) \psi,      \hat T = -\tfrac12 \partial_x^2,   \hat V = V(x).

The exact step \psi(t + \Delta t) = e^{-i(\hat T + \hat V)\Delta t} \psi(t) is hard because T and V do not
commute. Strang splitting approximates it (error O(\Delta t^3) per step) by

    e^{-i\hat V \Delta t/2} \; e^{-i\hat T \Delta t} \; e^{-i\hat V \Delta t/2}.

Each factor is DIAGONAL in its own basis, and a diagonal operator D_{xy} = d_x \delta_{xy} acts as
D_{xy}\psi_y = d_x \psi_x (no sum left): an elementwise multiply.

    potential, in x-space:   \psi_x \leftarrow P_x \psi_x,            P_x = e^{-i V_x \Delta t / 2}
    kinetic,   in k-space:   \tilde\psi_k \leftarrow K_k \tilde\psi_k,  K_k = e^{-i k^2 \Delta t / 2}

The basis change \tilde\psi_k = F_{kx}\psi_x (x summed) is the discrete Fourier transform, done by torch.fft.
"""

import torch


def potential_phase(V: torch.Tensor, dt: float) -> torch.Tensor:
    r"""P_x = e^{-i V_x \Delta t/2}.   V: (N,) → (N,) complex"""
    return torch.exp(-0.5j * V * dt)


def kinetic_phase(k: torch.Tensor, dt: float) -> torch.Tensor:
    r"""K_k = e^{-i (k^2/2) \Delta t}.   k: (N,) → (N,) complex"""
    return torch.exp(-0.5j * k**2 * dt)


def split_step(psi: torch.Tensor, P: torch.Tensor, K: torch.Tensor) -> torch.Tensor:
    r"""One Strang step  \psi \to P \, F^{-1} K F \, P \, \psi.   All tensors (N,) complex."""
    psi = P * psi  # half potential step, diagonal in x
    psi_k = torch.fft.fft(psi)  # x → k:  \tilde\psi_k = F_{kx} \psi_x
    psi_k = K * psi_k  # full kinetic step, diagonal in k
    psi = torch.fft.ifft(psi_k)  # k → x
    return P * psi  # second half potential step


def probability(psi: torch.Tensor, dx: float) -> torch.Tensor:
    r"""\int |\psi|^2 dx \approx \psi^*_x \psi_x dx  (x summed).   (..., N) → (...)"""
    return torch.einsum("...x,...x->...", psi.conj(), psi).real * dx


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/quantum/tunneling/tensors.py
"""Build the concrete tensors: periodic grid, its wavenumbers, the barrier and the initial wave packet."""

import math

import torch


def grid(n: int, length: float, device: torch.device) -> tuple[torch.Tensor, torch.Tensor, float]:
    """Periodic grid x ∈ [−length/2, length/2) and its FFT wavenumbers k. Returns (x (N,), k (N,), dx)."""
    dx = length / n
    x = -length / 2 + dx * torch.arange(n, dtype=torch.float64, device=device)  # (N,)
    k = 2 * math.pi * torch.fft.fftfreq(n, d=dx, dtype=torch.float64, device=device)  # (N,) in FFT order
    return x, k, dx


def barrier(x: torch.Tensor, V0: float, width: float) -> torch.Tensor:
    """V_x = V0 for |x| < a/2, else 0. Shape (N,)."""
    return torch.where(x.abs() < width / 2, V0, 0.0).to(x.dtype)


def grid_width(x: torch.Tensor, width: float, dx: float) -> float:
    """Width the barrier really has on the grid: a whole number of cells, so not exactly a."""
    return int((x.abs() < width / 2).sum()) * dx


def gaussian_packet(x: torch.Tensor, x0: float, sigma: float, k0: float, dx: float) -> torch.Tensor:
    """ψ_x = exp(−(x−x0)²/(4σ²) + i k0 x), normalised to Σ|ψ|² dx = 1. σ is the width of |ψ|². Shape (N,) complex."""
    psi = torch.exp(-((x - x0) ** 2) / (4 * sigma**2) + 1j * k0 * x)
    return psi / torch.sqrt((psi.abs() ** 2).sum() * dx)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/quantum/tunneling/solve.py
"""March the wave packet with split-step Fourier and measure what gets through the barrier."""

import math

import torch

from .equation import kinetic_phase, potential_phase, probability, split_step


def solve(
    psi0: torch.Tensor, V: torch.Tensor, k: torch.Tensor, t_max: float, dt: float = 0.01, n_saved: int = 240
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T,), ψ (T, N) complex), keeping about n_saved snapshots.

    P and K do not change in time, so they are built once; every step is two FFTs and three multiplies.
    """
    n_steps = int(round(t_max / dt))
    save_every = max(1, n_steps // n_saved)
    P = potential_phase(V, dt)  # (N,)
    K = kinetic_phase(k, dt)  # (N,)

    psi, snapshots, times = psi0, [psi0], [0.0]
    for step in range(1, n_steps + 1):
        psi = split_step(psi, P, K)
        if step % save_every == 0:
            snapshots.append(psi)
            times.append(step * dt)
    return torch.tensor(times), torch.stack(snapshots)


def transmission(psi: torch.Tensor, x: torch.Tensor, width: float, dx: float) -> torch.Tensor:
    """P_T: probability to the right of the barrier, for each snapshot. (..., N) → (...)"""
    right = (x > width / 2).to(psi.dtype)  # (N,) mask
    return probability(psi * right, dx)


def plane_wave_transmission(E: float, V0: float, width: float) -> float:
    """Textbook T for a plane wave of energy E on a barrier V0 of width a (ħ = m = 1)."""
    if V0 == 0:
        return 1.0
    if abs(E - V0) < 1e-9:
        return 1.0 / (1.0 + V0 * width**2 / 2)
    if E < V0:
        kappa = math.sqrt(2 * (V0 - E))
        return 1.0 / (1.0 + V0**2 * math.sinh(kappa * width) ** 2 / (4 * E * (V0 - E)))
    q = math.sqrt(2 * (E - V0))
    return 1.0 / (1.0 + V0**2 * math.sin(q * width) ** 2 / (4 * E * (E - V0)))


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/quantum/tunneling/plot.py
"""Animate |ψ|² hitting the barrier, with live transmitted / reflected probabilities."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(
    x: torch.Tensor,
    times: torch.Tensor,
    psi: torch.Tensor,
    V: torch.Tensor,
    E: float,
    p_trans: torch.Tensor,
    path: Path,
    x_view: float = 60.0,
) -> Path:
    xs = x.cpu().numpy()
    prob = (psi.abs() ** 2).cpu().numpy()  # (T, N)
    pt = p_trans.cpu().numpy()
    t = times.cpu().numpy()
    top = 1.15 * prob.max()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.set_xlim(-x_view, x_view)
    ax.set_ylim(0, top)
    ax.set_xlabel("x")
    ax.set_ylabel("|ψ|²")

    # Barrier and packet energy on a second axis in energy units.
    ax_v = ax.twinx()
    ax_v.grid(False)
    v_top = 1.6 * max(float(V.max()), E, 1e-9)
    ax_v.set_ylim(0, v_top)
    ax_v.set_ylabel("V(x),  E", color="#6b7280")
    ax_v.fill_between(xs, V.cpu().numpy(), color="#6b7280", alpha=0.35, step="mid", label="barrier V(x)")
    ax_v.axhline(E, color=ACCENT[4], ls="--", lw=1, label="E = k0²/2")
    ax_v.legend(loc="upper right", fontsize=8)

    (line,) = ax.plot([], [], color=ACCENT[0], lw=2)
    fill = [ax.fill_between(xs, 0 * xs, color=ACCENT[0], alpha=0.2)]
    label = ax.text(0.02, 0.95, "", transform=ax.transAxes, va="top", family="monospace")

    def update(step: int):
        line.set_data(xs, prob[step])
        fill[0].remove()
        fill[0] = ax.fill_between(xs, prob[step], color=ACCENT[0], alpha=0.2)
        label.set_text(f"t = {t[step]:5.1f}\nP(right) = {pt[step]:.3f}\nP(left)  = {1 - pt[step]:.3f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t)), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'k0': 2.0,
    'V0': 2.5,
    'width': 1.0,
    'sigma': 2.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.quantum.tunneling")]:
    del sys.modules[name]
main = importlib.import_module("problems.quantum.tunneling.main")
problem_dir = Path("problems/quantum/tunneling")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
